# MDCE Pit Proof Finalizer

This notebook does not rerun ingestion or retrain the heavy models. It packages the current pit proof layer into one final decision report.

It reads the outputs from the main notebook, hardening notebook, bias guard notebook, and model challenger notebook, then writes the selected model registry and final pit proof decision.

Expected selected model from current evidence: `rf_800_leaf5_half_balanced_subsample`.


## 1. Mount Drive

Colab requires mounting Drive. The code itself is locked to `/content/drive/MyDrive/ibm_project_stuff/MDCE` and does not scan MyDrive.


In [1]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


## 2. Install Dependencies


In [2]:
%pip install -q pandas


## 3. Run Finalizer

This creates the final model decision document and registry.


In [3]:
from pathlib import Path
import json

import pandas as pd


ROOT = Path("/content/drive/MyDrive/ibm_project_stuff/MDCE")
if not ROOT.exists():
    raise FileNotFoundError(
        f"Expected project folder not found: {ROOT}. "
        "This finalizer is folder-locked and will not scan MyDrive."
    )

REPORT_DIR = ROOT / "outputs" / "reports"
MODEL_DIR = ROOT / "outputs" / "models"
CHART_DIR = ROOT / "outputs" / "charts"

for folder in [REPORT_DIR, MODEL_DIR, CHART_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


def read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def read_csv(path):
    path = Path(path)
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


pit_metrics = read_json(REPORT_DIR / "pit_model_metrics.json").get("metrics", {})
event_metrics = read_json(REPORT_DIR / "ml_decision_event_metrics.json")
bias_guard = read_json(REPORT_DIR / "bias_guard_audit.json")
challenger = read_json(REPORT_DIR / "model_challenger_report.json")

model_challenger_df = read_csv(REPORT_DIR / "model_challenger_leaderboard.csv")
bias_feature_df = read_csv(REPORT_DIR / "bias_guard_feature_set_audit.csv")
hardening_df = read_csv(REPORT_DIR / "advanced_model_leaderboard.csv")
multi_horizon_df = read_csv(REPORT_DIR / "advanced_multi_horizon_metrics.csv")
robustness_df = read_csv(REPORT_DIR / "advanced_robustness_stress_tests.csv")
permutation_df = read_csv(REPORT_DIR / "advanced_permutation_importance.csv")
circuit_df = read_csv(REPORT_DIR / "bias_guard_circuit_classification_metrics.csv")
driver_df = read_csv(REPORT_DIR / "bias_guard_driver_classification_metrics.csv")

if model_challenger_df.empty:
    raise FileNotFoundError("Missing model_challenger_leaderboard.csv. Run MCDE_model_challenger_addon.ipynb first.")

selected = model_challenger_df.sort_values(["average_precision", "f1"], ascending=False).iloc[0].to_dict()
selected_model_path = (
    MODEL_DIR / "pit_window_challenger_best_model.pt"
    if str(selected["model_family"]) == "pytorch_mlp"
    else MODEL_DIR / "pit_window_challenger_best_model.joblib"
)
torch_model_path = MODEL_DIR / "pit_window_challenger_best_model.pt"

registry_rows = []
for _, row in model_challenger_df.iterrows():
    reason = "selected_best_holdout_ap_and_f1" if row["model"] == selected["model"] else "rejected_lower_holdout_score"
    if str(row.get("model_family")) == "pytorch_mlp" and row["model"] != selected["model"]:
        reason = "rejected_neural_net_underperformed_tree_ensemble_on_holdout"
    registry_rows.append(
        {
            "model": row["model"],
            "family": row["model_family"],
            "status": "selected" if row["model"] == selected["model"] else "rejected",
            "reason": reason,
            "average_precision": row["average_precision"],
            "roc_auc": row["roc_auc"],
            "f1": row["f1"],
            "precision": row["precision"],
            "recall": row["recall"],
            "brier_score": row["brier_score"],
            "threshold": row["threshold"],
        }
    )
registry_df = pd.DataFrame(registry_rows)
registry_df.to_csv(REPORT_DIR / "pit_final_model_registry.csv", index=False)

ap_baseline = pit_metrics.get("average_precision")
ap_selected = float(selected["average_precision"])
f1_baseline = pit_metrics.get("f1")
f1_selected = float(selected["f1"])

schedule_gap = bias_guard.get("full_vs_no_schedule_average_precision_gap")
best_bias_feature_set = bias_guard.get("best_feature_set")

weak_circuits = circuit_df.sort_values("f1", ascending=True).head(10) if not circuit_df.empty else pd.DataFrame()
weak_drivers = driver_df.sort_values("f1", ascending=True).head(10) if not driver_df.empty else pd.DataFrame()

final_proof = {
    "selected_model": {
        "name": selected["model"],
        "family": selected["model_family"],
        "path": str(selected_model_path),
        "artifact_exists": selected_model_path.exists(),
        "feature_set": "no_identity_context",
        "threshold": float(selected["threshold"]),
        "threshold_protocol": "selected on 2022 validation, evaluated on untouched 2023 holdout",
        "metrics_2023": {
            "average_precision": ap_selected,
            "roc_auc": float(selected["roc_auc"]),
            "f1": f1_selected,
            "precision": float(selected["precision"]),
            "recall": float(selected["recall"]),
            "brier_score": float(selected["brier_score"]),
        },
    },
    "improvement_over_initial_random_forest": {
        "initial_average_precision": ap_baseline,
        "selected_average_precision": ap_selected,
        "average_precision_delta": ap_selected - float(ap_baseline) if ap_baseline is not None else None,
        "initial_f1": f1_baseline,
        "selected_f1": f1_selected,
        "f1_delta": f1_selected - float(f1_baseline) if f1_baseline is not None else None,
    },
    "bias_guard": {
        "best_feature_set": best_bias_feature_set,
        "schedule_dependency_average_precision_gap": schedule_gap,
        "interpretation": "Lap/progress timing is an important known-at-decision-time signal. It is allowed but disclosed because it reflects typical strategy windows.",
    },
    "neural_network_decision": {
        "tested": True,
        "promoted": str(selected["model_family"]) == "pytorch_mlp",
        "torch_artifact_path": str(torch_model_path),
        "torch_artifact_exists": torch_model_path.exists(),
        "decision": "PyTorch challenger rejected because it did not beat the tree ensemble on 2023 holdout metrics." if str(selected["model_family"]) != "pytorch_mlp" else "PyTorch challenger selected by holdout evidence.",
    },
    "event_backtest": event_metrics,
    "known_weak_spots": {
        "circuits": weak_circuits.to_dict(orient="records"),
        "drivers": weak_drivers.to_dict(orient="records"),
    },
}
(REPORT_DIR / "pit_final_model_decision.json").write_text(json.dumps(final_proof, indent=2, default=str), encoding="utf-8")

lines = [
    "# MDCE Final Pit Proof Layer Decision",
    "",
    "## Selected Model",
    "",
    f"- Model: `{selected['model']}`",
    f"- Family: `{selected['model_family']}`",
    f"- Path: `{selected_model_path}`",
    "- Feature set: `no_identity_context`",
    f"- Threshold: `{selected['threshold']}`",
    "- Threshold protocol: selected on 2022 validation, evaluated on untouched 2023 holdout.",
    "",
    "## Final 2023 Holdout Metrics",
    "",
    f"- Average precision: `{float(selected['average_precision']):.3f}`",
    f"- ROC-AUC: `{float(selected['roc_auc']):.3f}`",
    f"- F1: `{float(selected['f1']):.3f}`",
    f"- Precision: `{float(selected['precision']):.3f}`",
    f"- Recall: `{float(selected['recall']):.3f}`",
    f"- Brier score: `{float(selected['brier_score']):.3f}`",
    "",
    "## Improvement Over Initial Model",
    "",
]
if ap_baseline is not None:
    lines += [
        f"- Initial AP: `{float(ap_baseline):.3f}`",
        f"- Final AP: `{ap_selected:.3f}`",
        f"- AP delta: `{ap_selected - float(ap_baseline):.3f}`",
        f"- Initial F1: `{float(f1_baseline):.3f}`",
        f"- Final F1: `{f1_selected:.3f}`",
        f"- F1 delta: `{f1_selected - float(f1_baseline):.3f}`",
    ]
else:
    lines.append("- Initial model metrics were not available.")

lines += [
    "",
    "## Model Challenger Outcome",
    "",
    "| Model | Family | Status | AP | ROC-AUC | F1 | Precision | Recall | Brier | Reason |",
    "|---|---|---|---:|---:|---:|---:|---:|---:|---|",
]
for _, row in registry_df.iterrows():
    lines.append(
        f"| {row['model']} | {row['family']} | {row['status']} | {row['average_precision']:.3f} | {row['roc_auc']:.3f} | {row['f1']:.3f} | {row['precision']:.3f} | {row['recall']:.3f} | {row['brier_score']:.3f} | {row['reason']} |"
    )

lines += [
    "",
    "## Neural Network Decision",
    "",
]
if str(selected["model_family"]) == "pytorch_mlp":
    lines.append("- PyTorch challenger won and is promoted by holdout evidence.")
else:
    lines.append("- PyTorch challenger was tested and rejected because it did not beat the tree ensemble on 2023 holdout metrics.")
    lines.append("- The `.pt` file being absent is expected in this run because the neural network was not promoted.")

lines += [
    "",
    "## Bias Guard Result",
    "",
    f"- Best feature set from bias guard: `{best_bias_feature_set}`",
    f"- Schedule/progress AP dependency gap: `{float(schedule_gap):.3f}`" if schedule_gap is not None else "- Schedule/progress AP dependency gap: unavailable",
    "- Interpretation: lap/progress timing is a strong known-at-decision-time signal. It is not hidden; the final proof layer discloses it.",
    "",
    "## Event Backtest",
    "",
]
if event_metrics:
    lines += [
        f"- Events tested: `{event_metrics.get('event_count')}`",
        f"- Mean absolute lap error: `{event_metrics.get('mean_abs_lap_error')}`",
        f"- Median absolute lap error: `{event_metrics.get('median_abs_lap_error')}`",
        f"- Within 1 lap rate: `{event_metrics.get('within_1_lap_rate')}`",
        f"- Within 3 laps rate: `{event_metrics.get('within_3_lap_rate')}`",
        f"- Within 5 laps rate: `{event_metrics.get('within_5_lap_rate')}`",
    ]
else:
    lines.append("- Event backtest metrics unavailable.")

lines += [
    "",
    "## Known Weak Circuit Groups",
    "",
]
if weak_circuits.empty:
    lines.append("- Circuit weak-spot metrics unavailable.")
else:
    for _, row in weak_circuits.iterrows():
        lines.append(f"- `{row['circuit']}`: F1 `{row['f1']:.3f}`, AP `{row['average_precision']:.3f}`, rows `{int(row['rows'])}`")

lines += [
    "",
    "## Safe Final Claim",
    "",
    "The final pit proof layer can support public-data pit-window confidence decisions. It does not claim to reproduce private F1 strategy systems or globally optimal strategy. It is strongest when used as a confidence/risk layer around a proposed race-strategy decision.",
    "",
    "## Files",
    "",
    f"- `{REPORT_DIR / 'pit_final_model_registry.csv'}`",
    f"- `{REPORT_DIR / 'pit_final_model_decision.json'}`",
    f"- `{selected_model_path}`",
]
(REPORT_DIR / "pit_final_model_decision.md").write_text("\n".join(lines), encoding="utf-8")

print("PIT PROOF FINALIZER COMPLETE")
print("Selected model:", selected["model"])
print("Selected model path:", selected_model_path)
print("Report:", REPORT_DIR / "pit_final_model_decision.md")


PIT PROOF FINALIZER COMPLETE
Selected model: rf_800_leaf5_half_balanced_subsample
Selected model path: /content/drive/MyDrive/ibm_project_stuff/MDCE/outputs/models/pit_window_challenger_best_model.joblib
Report: /content/drive/MyDrive/ibm_project_stuff/MDCE/outputs/reports/pit_final_model_decision.md


## 4. View Final Outputs


In [4]:
from pathlib import Path
from IPython.display import Markdown, display
import pandas as pd

ROOT = Path("/content/drive/MyDrive/ibm_project_stuff/MDCE")
REPORT_DIR = ROOT / "outputs" / "reports"
MODEL_DIR = ROOT / "outputs" / "models"

decision_md = REPORT_DIR / "pit_final_model_decision.md"
registry_csv = REPORT_DIR / "pit_final_model_registry.csv"
decision_json = REPORT_DIR / "pit_final_model_decision.json"
model_path = MODEL_DIR / "pit_window_challenger_best_model.joblib"

print("Decision report:", decision_md)
print("Registry:", registry_csv)
print("Decision JSON:", decision_json)
print("Selected model exists:", model_path.exists(), model_path)

display(Markdown(decision_md.read_text(encoding="utf-8")))
display(pd.read_csv(registry_csv))


Decision report: /content/drive/MyDrive/ibm_project_stuff/MDCE/outputs/reports/pit_final_model_decision.md
Registry: /content/drive/MyDrive/ibm_project_stuff/MDCE/outputs/reports/pit_final_model_registry.csv
Decision JSON: /content/drive/MyDrive/ibm_project_stuff/MDCE/outputs/reports/pit_final_model_decision.json
Selected model exists: True /content/drive/MyDrive/ibm_project_stuff/MDCE/outputs/models/pit_window_challenger_best_model.joblib


# MDCE Final Pit Proof Layer Decision

## Selected Model

- Model: `rf_800_leaf5_half_balanced_subsample`
- Family: `tree_ensemble`
- Path: `/content/drive/MyDrive/ibm_project_stuff/MDCE/outputs/models/pit_window_challenger_best_model.joblib`
- Feature set: `no_identity_context`
- Threshold: `0.5`
- Threshold protocol: selected on 2022 validation, evaluated on untouched 2023 holdout.

## Final 2023 Holdout Metrics

- Average precision: `0.820`
- ROC-AUC: `0.893`
- F1: `0.731`
- Precision: `0.874`
- Recall: `0.628`
- Brier score: `0.084`

## Improvement Over Initial Model

- Initial AP: `0.813`
- Final AP: `0.820`
- AP delta: `0.007`
- Initial F1: `0.719`
- Final F1: `0.731`
- F1 delta: `0.011`

## Model Challenger Outcome

| Model | Family | Status | AP | ROC-AUC | F1 | Precision | Recall | Brier | Reason |
|---|---|---|---:|---:|---:|---:|---:|---:|---|
| rf_800_leaf5_half_balanced_subsample | tree_ensemble | selected | 0.820 | 0.893 | 0.731 | 0.874 | 0.628 | 0.084 | selected_best_holdout_ap_and_f1 |
| rf_800_leaf3_sqrt_balanced | tree_ensemble | rejected | 0.819 | 0.895 | 0.723 | 0.791 | 0.667 | 0.085 | rejected_lower_holdout_score |
| rf_500_leaf2_sqrt_balanced | tree_ensemble | rejected | 0.817 | 0.894 | 0.725 | 0.875 | 0.620 | 0.085 | rejected_lower_holdout_score |
| torch_mlp_128_64_dropout20 | pytorch_mlp | rejected | 0.783 | 0.877 | 0.672 | 0.664 | 0.679 | 0.150 | rejected_neural_net_underperformed_tree_ensemble_on_holdout |
| torch_mlp_256_128_dropout25 | pytorch_mlp | rejected | 0.768 | 0.869 | 0.655 | 0.612 | 0.704 | 0.149 | rejected_neural_net_underperformed_tree_ensemble_on_holdout |
| extra_trees_600_leaf2_sqrt_balanced | tree_ensemble | rejected | 0.768 | 0.869 | 0.666 | 0.685 | 0.648 | 0.108 | rejected_lower_holdout_score |

## Neural Network Decision

- PyTorch challenger was tested and rejected because it did not beat the tree ensemble on 2023 holdout metrics.
- The `.pt` file being absent is expected in this run because the neural network was not promoted.

## Bias Guard Result

- Best feature set from bias guard: `no_identity_context`
- Schedule/progress AP dependency gap: `0.211`
- Interpretation: lap/progress timing is a strong known-at-decision-time signal. It is not hidden; the final proof layer discloses it.

## Event Backtest

- Events tested: `417`
- Mean absolute lap error: `1.5107913669064748`
- Median absolute lap error: `0.0`
- Within 1 lap rate: `0.6163069544364509`
- Within 3 laps rate: `0.8321342925659473`
- Within 5 laps rate: `0.9448441247002398`

## Known Weak Circuit Groups

- `Circuit Park Zandvoort`: F1 `0.578`, AP `0.683`, rows `1162`
- `Silverstone Circuit`: F1 `0.631`, AP `0.881`, rows `562`
- `Red Bull Ring`: F1 `0.639`, AP `0.780`, rows `845`
- `Albert Park Grand Prix Circuit`: F1 `0.656`, AP `0.706`, rows `920`
- `Losail International Circuit`: F1 `0.672`, AP `0.829`, rows `762`
- `Autódromo José Carlos Pace`: F1 `0.688`, AP `0.838`, rows `784`
- `Circuit de Spa-Francorchamps`: F1 `0.696`, AP `0.827`, rows `483`
- `Suzuka Circuit`: F1 `0.700`, AP `0.806`, rows `545`
- `Circuit Gilles Villeneuve`: F1 `0.703`, AP `0.791`, rows `696`
- `Circuit de Monaco`: F1 `0.714`, AP `0.828`, rows `1093`

## Safe Final Claim

The final pit proof layer can support public-data pit-window confidence decisions. It does not claim to reproduce private F1 strategy systems or globally optimal strategy. It is strongest when used as a confidence/risk layer around a proposed race-strategy decision.

## Files

- `/content/drive/MyDrive/ibm_project_stuff/MDCE/outputs/reports/pit_final_model_registry.csv`
- `/content/drive/MyDrive/ibm_project_stuff/MDCE/outputs/reports/pit_final_model_decision.json`
- `/content/drive/MyDrive/ibm_project_stuff/MDCE/outputs/models/pit_window_challenger_best_model.joblib`

,model,family,status,reason,average_precision,roc_auc,f1,precision,recall,brier_score,threshold
0,rf_800_leaf5_half_balanced_subsample,tree_ensemble,selected,selected_best_holdout_ap_and_f1,0.819995,0.893386,0.730783,0.873878,0.627957,0.083661,0.50
1,rf_800_leaf3_sqrt_balanced,tree_ensemble,rejected,rejected_lower_holdout_score,0.819363,0.894588,0.723333,0.790528,0.666667,0.085164,0.40
2,rf_500_leaf2_sqrt_balanced,tree_ensemble,rejected,rejected_lower_holdout_score,0.817363,0.893556,0.725409,0.874675,0.619662,0.084675,0.45
3,torch_mlp_128_64_dropout20,pytorch_mlp,rejected,rejected_neural_net_underperformed_tree_ensemb...,0.782980,0.876728,0.671630,0.664462,0.678955,0.150246,0.80
4,torch_mlp_256_128_dropout25,pytorch_mlp,rejected,rejected_neural_net_underperformed_tree_ensemb...,0.767686,0.869497,0.654577,0.611526,0.704147,0.148557,0.70
5,extra_trees_600_leaf2_sqrt_balanced,tree_ensemble,rejected,rejected_lower_holdout_score,0.767571,0.868621,0.666035,0.684843,0.648233,0.108297,0.45


## 5. What To Send Back

Send these outputs back in chat:

- The `PIT PROOF FINALIZER COMPLETE` block.
- The selected model line.
- The displayed final Markdown report.
- Any error traceback if the notebook stops.
